## APP Assignment – Solutions (Problems 1–65)

## 1. Single Number

In [1]:
def single_number(nums):
    """XOR of all numbers: pairs cancel out (a ^ a = 0), the single one remains. O(n) time, O(1) space."""
    result = 0
    for x in nums:
        result ^= x
    return result

for nums in [[2, 2, 1], [4, 1, 2, 1, 2], [1]]:
    print(f"Input: nums = {nums}  ->  Output: {single_number(nums)}")

Input: nums = [2, 2, 1]  ->  Output: 1
Input: nums = [4, 1, 2, 1, 2]  ->  Output: 4
Input: nums = [1]  ->  Output: 1


## 2. Find Palindromes in a Given Range

In [2]:
def palindromes_in_range(m, n):
    return [x for x in range(m, n + 1) if str(x) == str(x)[::-1]]

for m, n in [(10, 50), (100, 150), (1, 9)]:
    print(f"Input: m = {m}, n = {n}  ->  Output: {palindromes_in_range(m, n)}")

Input: m = 10, n = 50  ->  Output: [11, 22, 33, 44]
Input: m = 100, n = 150  ->  Output: [101, 111, 121, 131, 141]
Input: m = 1, n = 9  ->  Output: [1, 2, 3, 4, 5, 6, 7, 8, 9]


## 3. Regular Expression Matching

In [3]:
def is_match(s, p):
    """dp[i][j] = True if s[:i] matches p[:j]."""
    m, n = len(s), len(p)
    dp = [[False] * (n + 1) for _ in range(m + 1)]
    dp[0][0] = True
    for j in range(2, n + 1):                       # patterns like a*, a*b* can match an empty string
        if p[j - 1] == "*":
            dp[0][j] = dp[0][j - 2]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if p[j - 1] == "*":
                dp[i][j] = dp[i][j - 2]             # use '*' as zero occurrences
                if p[j - 2] == "." or p[j - 2] == s[i - 1]:
                    dp[i][j] = dp[i][j] or dp[i - 1][j]   # one more occurrence
            elif p[j - 1] == "." or p[j - 1] == s[i - 1]:
                dp[i][j] = dp[i - 1][j - 1]
    return dp[m][n]

for s, p in [("abcd", ".*"), ("mississippi", "misisp*."), ("aa", "a")]:
    print(f'Input: s = "{s}", p = "{p}"  ->  Output: {str(is_match(s, p)).lower()}')

Input: s = "abcd", p = ".*"  ->  Output: true
Input: s = "mississippi", p = "misisp*."  ->  Output: false
Input: s = "aa", p = "a"  ->  Output: false


## 4. Decode Ways with Pattern Generation

In [4]:
def decode_ways(s):
    # Count with dynamic programming
    n = len(s)
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        if s[i - 1] != "0":
            dp[i] += dp[i - 1]
        if i >= 2 and s[i - 2] != "0" and 10 <= int(s[i - 2:i]) <= 26:
            dp[i] += dp[i - 2]

    # List every decoding with backtracking
    patterns = []
    def backtrack(i, current):
        if i == n:
            patterns.append(current); return
        if s[i] == "0":
            return                                   # no segment may start with 0
        backtrack(i + 1, current + chr(ord("A") + int(s[i]) - 1))
        if i + 1 < n and 10 <= int(s[i:i + 2]) <= 26:
            backtrack(i + 2, current + chr(ord("A") + int(s[i:i + 2]) - 1))
    backtrack(0, "")
    return dp[n], patterns

for s in ["12", "226", "11106", "06"]:
    total, pats = decode_ways(s)
    print(f'Input: s = "{s}"')
    print("Total Ways:", total)
    print("Decoded Strings:")
    print("\n".join(pats) if pats else "None")
    print()

Input: s = "12"
Total Ways: 2
Decoded Strings:
AB
L

Input: s = "226"
Total Ways: 3
Decoded Strings:
BBF
BZ
VF

Input: s = "11106"
Total Ways: 2
Decoded Strings:
AAJF
KJF

Input: s = "06"
Total Ways: 0
Decoded Strings:
None



## 5. Balanced Binary Tree

In [5]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build_tree(values):
    """Build a binary tree from a level-order list (None = missing node)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); q.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); q.append(node.right)
        i += 1
    return root

def is_balanced(root):
    def height(node):                 # returns -1 as soon as a subtree is unbalanced
        if not node: return 0
        lh = height(node.left)
        if lh == -1: return -1
        rh = height(node.right)
        if rh == -1 or abs(lh - rh) > 1: return -1
        return 1 + max(lh, rh)
    return height(root) != -1

for vals in [[3, 9, 20, None, None, 15, 7], [1, 2, 2, 3, 3, None, None, 4, 4], [4]]:
    print(f"Input: root = {vals}  ->  Output: {str(is_balanced(build_tree(vals))).lower()}")

Input: root = [3, 9, 20, None, None, 15, 7]  ->  Output: true
Input: root = [1, 2, 2, 3, 3, None, None, 4, 4]  ->  Output: false
Input: root = [4]  ->  Output: true


## 6. Reverse Integer

In [6]:
INT_MAX, INT_MIN = 2**31 - 1, -2**31

def reverse_integer(x):
    """Builds the result digit by digit and checks for overflow *before* each step,
    so no value outside the 32-bit range is ever stored."""
    sign = -1 if x < 0 else 1
    x = abs(x)
    result = 0
    while x:
        digit = x % 10; x //= 10
        if result > (INT_MAX - digit) // 10:     # result*10 + digit would overflow
            return 0
        result = result * 10 + digit
    result *= sign
    return result if INT_MIN <= result <= INT_MAX else 0

for x in [123, -123, 120, 1534236469]:
    print(f"Input: x = {x}  ->  Output: {reverse_integer(x)}")

Input: x = 123  ->  Output: 321
Input: x = -123  ->  Output: -321
Input: x = 120  ->  Output: 21
Input: x = 1534236469  ->  Output: 0


## 7. Minimize Maximum Consecutive Zeros with Construction

In [7]:
from itertools import combinations

def max_zero_block(s):
    return max((len(block) for block in s.split("1")), default=0)

def min_max_zero_arrangements(L, K):
    strings = []
    for ones in combinations(range(L), K):          # every way to place K ones among L positions
        s = ["0"] * L
        for i in ones: s[i] = "1"
        strings.append("".join(s))
    scored = [(s, max_zero_block(s)) for s in strings]
    best = min(score for _, score in scored)
    return scored, [s for s, score in scored if score == best], best

for L, K in [(4, 2), (5, 0), (5, 5)]:
    scored, optimal, best = min_max_zero_arrangements(L, K)
    print(f"Input: {L} {K}")
    print("Possible Strings:")
    for s, score in scored:
        print(f"{s} → max zeros = {score}")
    print("Optimal Arrangement(s):", ", ".join(optimal))
    print("Minimum Longest Zero Block:", best)
    print()

Input: 4 2
Possible Strings:
1100 → max zeros = 2
1010 → max zeros = 1
1001 → max zeros = 2
0110 → max zeros = 1
0101 → max zeros = 1
0011 → max zeros = 2
Optimal Arrangement(s): 1010, 0110, 0101
Minimum Longest Zero Block: 1

Input: 5 0
Possible Strings:
00000 → max zeros = 5
Optimal Arrangement(s): 00000
Minimum Longest Zero Block: 5

Input: 5 5
Possible Strings:
11111 → max zeros = 0
Optimal Arrangement(s): 11111
Minimum Longest Zero Block: 0



## 8. Merge Triplets to Form Target Triplet

In [8]:
def merge_triplets(triplets, target):
    """Only triplets with every value <= the target can be merged without overshooting.
    The target is reachable if those 'good' triplets together reach each target value."""
    good = [t for t in triplets if all(t[i] <= target[i] for i in range(3))]
    reached = [False, False, False]
    used = []
    for t in good:
        contributes = False
        for i in range(3):
            if t[i] == target[i] and not reached[i]:
                reached[i] = True; contributes = True
        if contributes: used.append(t)
    return all(reached), used

examples = [([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5]),
            ([[3, 4, 5], [4, 5, 6]], [3, 2, 5]),
            ([[2, 5, 3], [2, 3, 4], [1, 2, 5], [5, 2, 3]], [5, 5, 5])]
for triplets, target in examples:
    ok, used = merge_triplets(triplets, target)
    print(f"Input: triplets = {triplets}, target = {target}")
    print("Output:", str(ok).lower())
    print("Triplets merged to reach the target:", used if ok else "none")
    print("Target =", target if ok else "none")
    print()

Input: triplets = [[2, 5, 3], [1, 8, 4], [1, 7, 5]], target = [2, 7, 5]
Output: true
Triplets merged to reach the target: [[2, 5, 3], [1, 7, 5]]
Target = [2, 7, 5]

Input: triplets = [[3, 4, 5], [4, 5, 6]], target = [3, 2, 5]
Output: false
Triplets merged to reach the target: none
Target = none

Input: triplets = [[2, 5, 3], [2, 3, 4], [1, 2, 5], [5, 2, 3]], target = [5, 5, 5]
Output: true
Triplets merged to reach the target: [[2, 5, 3], [1, 2, 5], [5, 2, 3]]
Target = [5, 5, 5]



## 9. Palindromic Substrings with Pattern Listing

In [9]:
def palindromic_substrings(s):
    found = []                                     # (length, start, text)
    for center in range(2 * len(s) - 1):           # expand around every centre (character or gap)
        left, right = center // 2, center // 2 + center % 2
        while left >= 0 and right < len(s) and s[left] == s[right]:
            found.append((right - left + 1, left, s[left:right + 1]))
            left -= 1; right += 1
    found.sort()                                   # by length, then by position
    return [text for _, _, text in found]

for s in ["ababa", "racecar", "level"]:
    pals = palindromic_substrings(s)
    print(f'Input: s = "{s}"')
    print("Total Count:", len(pals))
    print("Palindromic Substrings:")
    print("\n".join(pals))
    print()

Input: s = "ababa"
Total Count: 9
Palindromic Substrings:
a
b
a
b
a
aba
bab
aba
ababa

Input: s = "racecar"
Total Count: 10
Palindromic Substrings:
r
a
c
e
c
a
r
cec
aceca
racecar

Input: s = "level"
Total Count: 7
Palindromic Substrings:
l
e
v
e
l
eve
level



## 10. Kth Smallest Element in a BST


In [10]:
def kth_smallest(root, k):
    """Iterative in-order traversal visits BST values in ascending order."""
    stack, node, count = [], root, 0
    while stack or node:
        while node:
            stack.append(node); node = node.left
        node = stack.pop()
        count += 1
        if count == k:
            return node.val
        node = node.right

for vals, k in [([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13], 5), ([5, 3, 6, 2, 4, None, None, 1], 3)]:
    print(f"Input: root = {vals}, k = {k}  ->  Output: {kth_smallest(build_tree(vals), k)}")

Input: root = [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13], k = 5  ->  Output: 7
Input: root = [5, 3, 6, 2, 4, None, None, 1], k = 3  ->  Output: 3


## 11. Missing Number

In [11]:
def missing_number(nums):
    n = len(nums)
    return n * (n + 1) // 2 - sum(nums)            # expected sum of 0..n minus actual sum

for nums in [[3, 0, 1], [0, 1], [9, 6, 4, 2, 3, 5, 7, 0, 1]]:
    print(f"Input: nums = {nums}  ->  Output: {missing_number(nums)}")

Input: nums = [3, 0, 1]  ->  Output: 2
Input: nums = [0, 1]  ->  Output: 2
Input: nums = [9, 6, 4, 2, 3, 5, 7, 0, 1]  ->  Output: 8


## 12. Coin Change

In [12]:
def coin_change(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount                       # dp[a] = fewest coins for amount a
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and dp[a - c] + 1 < dp[a]:
                dp[a] = dp[a - c] + 1
    return dp[amount] if dp[amount] != INF else -1

for coins, amount in [([1, 2, 5], 11), ([2], 3), ([1], 0)]:
    print(f"Input: coins = {coins}, amount = {amount}  ->  Output: {coin_change(coins, amount)}")

Input: coins = [1, 2, 5], amount = 11  ->  Output: 3
Input: coins = [2], amount = 3  ->  Output: -1
Input: coins = [1], amount = 0  ->  Output: 0


## 13. Cheapest Flights Within K Stops

In [13]:
def find_cheapest_price(n, flights, src, dst, k):
    """Bellman-Ford limited to k+1 edges (k stops)."""
    INF = float("inf")
    cost = [INF] * n
    cost[src] = 0
    for _ in range(k + 1):
        new_cost = cost[:]                          # use last round's costs only
        for u, v, price in flights:
            if cost[u] + price < new_cost[v]:
                new_cost[v] = cost[u] + price
        cost = new_cost
    return cost[dst] if cost[dst] != INF else -1

examples = [(4, [[0, 1, 100], [1, 2, 100], [2, 0, 100], [1, 3, 600], [2, 3, 200]], 0, 3, 1),
            (3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 1),
            (3, [[0, 1, 100], [1, 2, 100], [0, 2, 500]], 0, 2, 0)]
for n, flights, src, dst, k in examples:
    print(f"Input: n = {n}, src = {src}, dst = {dst}, k = {k}  ->  Output: {find_cheapest_price(n, flights, src, dst, k)}")

Input: n = 4, src = 0, dst = 3, k = 1  ->  Output: 700
Input: n = 3, src = 0, dst = 2, k = 1  ->  Output: 200
Input: n = 3, src = 0, dst = 2, k = 0  ->  Output: 500


## 14. Merge k Sorted Lists

In [14]:
import heapq

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

def to_linked(values):
    head = None
    for v in reversed(values): head = ListNode(v, head)
    return head

def to_list(head):
    out = []
    while head: out.append(head.val); head = head.next
    return out

def merge_k_lists(lists):
    """Min-heap holds the current head of each list. O(N log k)."""
    heap = [(node.val, i, node) for i, node in enumerate(lists) if node]
    heapq.heapify(heap)
    dummy = tail = ListNode()
    while heap:
        val, i, node = heapq.heappop(heap)
        tail.next = node; tail = node
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

for lists in [[[1, 4, 5], [1, 3, 4], [2, 6]], [[4, 5], [1, 6]]]:
    merged = merge_k_lists([to_linked(l) for l in lists])
    print(f"Input: lists = {lists}  ->  Output: {to_list(merged)}")

Input: lists = [[1, 4, 5], [1, 3, 4], [2, 6]]  ->  Output: [1, 1, 2, 3, 4, 4, 5, 6]
Input: lists = [[4, 5], [1, 6]]  ->  Output: [1, 4, 5, 6]


## 15. Reverse Bits

In [15]:
def reverse_bits(n):
    result = 0
    for _ in range(32):
        result = (result << 1) | (n & 1)            # take the lowest bit of n, append it to result
        n >>= 1
    return result

for n in [43261596, 2147483644]:
    r = reverse_bits(n)
    print(f"Input: n = {n} ({n:032b})  ->  Output: {r} ({r:032b})")

Input: n = 43261596 (00000010100101000001111010011100)  ->  Output: 964176192 (00111001011110000010100101000000)
Input: n = 2147483644 (01111111111111111111111111111100)  ->  Output: 1073741822 (00111111111111111111111111111110)


## 16. Separate Letters and Numbers from a String

In [16]:
def separate(s):
    letters = "".join(ch for ch in s if ch.isalpha())
    digits = "".join(ch for ch in s if ch.isdigit())
    return [letters, digits]

for s in ["a1b2c3d4", "x9y8z7", "12345", "abcdef"]:
    print(f'Input: s = "{s}"  ->  Output: {separate(s)}')

Input: s = "a1b2c3d4"  ->  Output: ['abcd', '1234']
Input: s = "x9y8z7"  ->  Output: ['xyz', '987']
Input: s = "12345"  ->  Output: ['', '12345']
Input: s = "abcdef"  ->  Output: ['abcdef', '']


## 17. Merge Intervals

In [17]:
def merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:        # overlaps (touching counts) with the last one
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

for intervals in [[[1, 3], [2, 6], [8, 10], [15, 18]], [[1, 4], [4, 5]], [[4, 7], [1, 4]]]:
    print(f"Input: intervals = {intervals}  ->  Output: {merge_intervals(intervals)}")

Input: intervals = [[1, 3], [2, 6], [8, 10], [15, 18]]  ->  Output: [[1, 6], [8, 10], [15, 18]]
Input: intervals = [[1, 4], [4, 5]]  ->  Output: [[1, 5]]
Input: intervals = [[4, 7], [1, 4]]  ->  Output: [[1, 7]]


## 18. Maximum Product Subarray

In [18]:
def max_product(nums):
    """Track the largest and smallest product ending here (a negative number swaps them)."""
    best = cur_max = cur_min = nums[0]
    for x in nums[1:]:
        if x < 0:
            cur_max, cur_min = cur_min, cur_max
        cur_max = max(x, cur_max * x)
        cur_min = min(x, cur_min * x)
        best = max(best, cur_max)
    return best

for nums in [[2, 3, -2, 4], [-2, 0, -1]]:
    print(f"Input: nums = {nums}  ->  Output: {max_product(nums)}")

Input: nums = [2, 3, -2, 4]  ->  Output: 6
Input: nums = [-2, 0, -1]  ->  Output: 0


## 19. Minimum Team Selection to Cover Required Skills

In [19]:
def smallest_team(required, candidates):
    """Bitmask dynamic programming: dp[mask] = smallest team covering exactly the skills in mask."""
    index = {skill: i for i, skill in enumerate(required)}
    full = (1 << len(required)) - 1
    masks = [sum(1 << index[s] for s in skills if s in index) for skills in candidates]
    dp = {0: []}
    for i, cmask in enumerate(masks):
        for mask, team in list(dp.items()):
            new = mask | cmask
            if new not in dp or len(team) + 1 < len(dp[new]):
                dp[new] = team + [i]
    return dp[full]

examples = [
    ("Python SQL MachineLearning DataStructures",
     ["Python SQL", "SQL MachineLearning", "MachineLearning DataStructures", "DataStructures"]),
    ("Java Python SQL", ["Java", "Python SQL", "SQL"]),
    ("C++ Python DataStructures", ["C++", "Python", "DataStructures", "C++ Python DataStructures"]),
]
for skills, cands in examples:
    team = smallest_team(skills.split(), [c.split() for c in cands])
    print("Skills:", skills)
    for i, c in enumerate(cands): print(f"  {i} → {c}")
    print("Output:", " ".join(map(str, team)))
    print()

Skills: Python SQL MachineLearning DataStructures
  0 → Python SQL
  1 → SQL MachineLearning
  2 → MachineLearning DataStructures
  3 → DataStructures
Output: 0 2

Skills: Java Python SQL
  0 → Java
  1 → Python SQL
  2 → SQL
Output: 0 1

Skills: C++ Python DataStructures
  0 → C++
  1 → Python
  2 → DataStructures
  3 → C++ Python DataStructures
Output: 3



## 20. Word Break with All Segmentations

In [20]:
from functools import lru_cache

def word_break(s, word_dict):
    words = set(word_dict)
    max_len = max(map(len, words))

    @lru_cache(maxsize=None)
    def segment(start):
        """All segmentations of s[start:] (longer words tried first)."""
        if start == len(s):
            return [[]]
        result = []
        for end in range(min(len(s), start + max_len), start, -1):
            word = s[start:end]
            if word in words:
                for rest in segment(end):
                    result.append([word] + rest)
        return result

    return [" ".join(seg) for seg in segment(0)]

examples = [("AmritaMysuru", ["Amrita", "Rita", "Mysuru"]),
            ("applepenapple", ["apple", "pen"]),
            ("catsanddog", ["cat", "cats", "and", "sand", "dog"]),
            ("catsandog", ["cats", "dog", "sand", "and", "cat"])]
for s, wd in examples:
    segs = word_break(s, wd)
    print(f'Input: s = "{s}", wordDict = {wd}')
    print("Can Segment:", str(bool(segs)).lower())
    print("Segmentations:", "\n".join(segs) if segs else "None")
    print()

Input: s = "AmritaMysuru", wordDict = ['Amrita', 'Rita', 'Mysuru']
Can Segment: true
Segmentations: Amrita Mysuru

Input: s = "applepenapple", wordDict = ['apple', 'pen']
Can Segment: true
Segmentations: apple pen apple

Input: s = "catsanddog", wordDict = ['cat', 'cats', 'and', 'sand', 'dog']
Can Segment: true
Segmentations: cats and dog
cat sand dog

Input: s = "catsandog", wordDict = ['cats', 'dog', 'sand', 'and', 'cat']
Can Segment: false
Segmentations: None



## 21. Number of 1 Bits

In [21]:
def hamming_weight(n):
    count = 0
    while n:
        n &= n - 1                                  # clears the lowest set bit
        count += 1
    return count

for n in [11, 128, 2147483645]:
    print(f"Input: n = {n} ({n:b})  ->  Output: {hamming_weight(n)}")

Input: n = 11 (1011)  ->  Output: 3
Input: n = 128 (10000000)  ->  Output: 1
Input: n = 2147483645 (1111111111111111111111111111101)  ->  Output: 30


## 22. Midpoint of Line Segment

In [22]:
def midpoint(p1, p2):
    mx, my = (p1[0] + p2[0]) / 2, (p1[1] + p2[1]) / 2
    tidy = lambda v: int(v) if v == int(v) else v   # show 2 instead of 2.0
    return (tidy(mx), tidy(my))

for p1, p2 in [((0, 0), (4, 4)), ((1, 3), (5, 7)), ((-2, 4), (2, 0))]:
    print(f"Point 1: {p1}, Point 2: {p2}  ->  Output: {midpoint(p1, p2)}")

Point 1: (0, 0), Point 2: (4, 4)  ->  Output: (2, 2)
Point 1: (1, 3), Point 2: (5, 7)  ->  Output: (3, 5)
Point 1: (-2, 4), Point 2: (2, 0)  ->  Output: (0, 2)


## 23. Sum of Cubes in a Given Range

In [23]:
def sum_of_cubes(a, b):
    """Uses 1³ + 2³ + ... + n³ = (n(n+1)/2)², so the answer is S(b) - S(a-1)."""
    S = lambda n: (n * (n + 1) // 2) ** 2
    return S(b) - S(a - 1)

for a, b in [(4, 9), (1, 3), (5, 5)]:
    print(f"Input: {a} {b}  ->  Output: {sum_of_cubes(a, b)}")

Input: 4 9  ->  Output: 1989
Input: 1 3  ->  Output: 36
Input: 5 5  ->  Output: 125


## 24. Closest Pair of Points

In [24]:
import math

def closest_pair(points):
    """Divide and conquer on squared distances, O(n log n)."""
    pts = sorted(points)
    def d2(p, q): return (p[0] - q[0]) ** 2 + (p[1] - q[1]) ** 2
    def solve(P):
        n = len(P)
        if n <= 3:
            return min(d2(P[i], P[j]) for i in range(n) for j in range(i + 1, n))
        mid = n // 2; mx = P[mid][0]
        best = min(solve(P[:mid]), solve(P[mid:]))
        strip = sorted((p for p in P if (p[0] - mx) ** 2 < best), key=lambda p: p[1])
        for i in range(len(strip)):
            for j in range(i + 1, len(strip)):
                if (strip[j][1] - strip[i][1]) ** 2 >= best: break
                best = min(best, d2(strip[i], strip[j]))
        return best
    best2 = solve(pts)
    root = math.isqrt(best2)
    exact = str(root) if root * root == best2 else f"√{best2}"
    return math.sqrt(best2), exact

for pts in [[(0, 0), (1, 1)], [(1, 1), (2, 2), (5, 5)], [(0, 0), (3, 4), (6, 8)]]:
    value, exact = closest_pair(pts)
    print(f"Input: points = {pts}  ->  Output: {exact} (≈ {value:.4f})")

Input: points = [(0, 0), (1, 1)]  ->  Output: √2 (≈ 1.4142)
Input: points = [(1, 1), (2, 2), (5, 5)]  ->  Output: √2 (≈ 1.4142)
Input: points = [(0, 0), (3, 4), (6, 8)]  ->  Output: 5 (≈ 5.0000)


## 25. Count Trailing Zeros in Binary

In [25]:
def trailing_zeros(n):
    if n == 0:
        return 0
    count = 0
    while n & 1 == 0:                               # look at the rightmost bit until a 1 appears
        n >>= 1; count += 1
    return count

for n in [8, 12]:
    print(f"Input: n = {n} (binary {n:b})  ->  Output: {trailing_zeros(n)}")

Input: n = 8 (binary 1000)  ->  Output: 3
Input: n = 12 (binary 1100)  ->  Output: 2


## 26. Sum of Unique Elements

In [26]:
from collections import Counter

def sum_unique(arr):
    counts = Counter(arr)
    unique = [x for x in arr if counts[x] == 1]
    return unique, sum(unique)

for arr in [[1, 3, 2, 3, 2, 4], [10, 20, 10, 30, 40, 30], [5, 5, 5, 5]]:
    unique, total = sum_unique(arr)
    print(f"Input: array = {arr}")
    print("Unique Elements:", " ".join(map(str, unique)) if unique else "None")
    print("Sum:", total)
    print()

Input: array = [1, 3, 2, 3, 2, 4]
Unique Elements: 1 4
Sum: 5

Input: array = [10, 20, 10, 30, 40, 30]
Unique Elements: 20 40
Sum: 60

Input: array = [5, 5, 5, 5]
Unique Elements: None
Sum: 0



## 27. Invert Binary Tree

In [27]:
def invert_tree(root):
    if root:
        root.left, root.right = invert_tree(root.right), invert_tree(root.left)
    return root

def level_order(root):
    out, q = [], deque([root] if root else [])
    while q:
        node = q.popleft(); out.append(node.val)
        if node.left: q.append(node.left)
        if node.right: q.append(node.right)
    return out

for vals in [[4, 2, 7, 1, 3, 6, 9], [2, 1, 3]]:
    print(f"Input: root = {vals}  ->  Output: {level_order(invert_tree(build_tree(vals)))}")

Input: root = [4, 2, 7, 1, 3, 6, 9]  ->  Output: [4, 7, 2, 9, 6, 3, 1]
Input: root = [2, 1, 3]  ->  Output: [2, 3, 1]


## 28. Valid Anagram

In [28]:
from collections import Counter

def is_anagram(s, t):
    return len(s) == len(t) and Counter(s) == Counter(t)

for s, t in [("anagram", "nagaram"), ("rat", "car"), ("listen", "silent"), ("hello", "helloo")]:
    print(f'Input: s = "{s}", t = "{t}"  ->  Output: {str(is_anagram(s, t)).lower()}')

Input: s = "anagram", t = "nagaram"  ->  Output: true
Input: s = "rat", t = "car"  ->  Output: false
Input: s = "listen", t = "silent"  ->  Output: true
Input: s = "hello", t = "helloo"  ->  Output: false


## 29. Happy Number

In [29]:
def is_happy(n):
    seen = set()
    while n != 1 and n not in seen:                 # a repeated number means an endless cycle
        seen.add(n)
        n = sum(int(d) ** 2 for d in str(n))
    return n == 1

for n in [19, 2]:
    print(f"Input: n = {n}  ->  Output: {str(is_happy(n)).lower()}")

Input: n = 19  ->  Output: true
Input: n = 2  ->  Output: false


## 30. Set Matrix Zeroes

In [30]:
def set_zeroes(matrix):
    """In place with O(1) extra space: the first row and column store the zero markers."""
    m, n = len(matrix), len(matrix[0])
    first_row_zero = any(matrix[0][j] == 0 for j in range(n))
    first_col_zero = any(matrix[i][0] == 0 for i in range(m))
    for i in range(1, m):
        for j in range(1, n):
            if matrix[i][j] == 0:
                matrix[i][0] = matrix[0][j] = 0
    for i in range(1, m):
        for j in range(1, n):
            if matrix[i][0] == 0 or matrix[0][j] == 0:
                matrix[i][j] = 0
    if first_row_zero:
        for j in range(n): matrix[0][j] = 0
    if first_col_zero:
        for i in range(m): matrix[i][0] = 0
    return matrix

for mat in [[[1, 1, 1], [1, 0, 1], [1, 1, 1]], [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]]:
    print(f"Input: matrix = {mat}", end="  ->  ")
    print(f"Output: {set_zeroes(mat)}")

Input: matrix = [[1, 1, 1], [1, 0, 1], [1, 1, 1]]  ->  Output: [[1, 0, 1], [0, 0, 0], [1, 0, 1]]
Input: matrix = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]  ->  Output: [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]


## 31. Card Probability Analysis

In [31]:
def card_probabilities(cards):
    n = len(cards)
    rank = lambda c: c[:-1]
    suit = lambda c: c[-1]
    counts = {"Ace": sum(rank(c) == "A" for c in cards),
              "Face": sum(rank(c) in ("J", "Q", "K") for c in cards),
              "Red": sum(suit(c) in ("H", "D") for c in cards)}
    black = sum(suit(c) in ("S", "C") for c in cards)
    probs = {k: v / n for k, v in counts.items()}
    most = max(counts, key=counts.get)              # ties: first in the order Ace, Face, Red
    return probs, most, black / n

def fmt(p):
    return f"{p:.2f}" if round(p, 2) == p else f"{p:.3f}"

cards = ["AH", "10D", "KC", "QS", "2H", "JD", "7C", "KH"]
probs, most, p_black = card_probabilities(cards)
print("Input: cards =", cards)
print(f"P(Ace)={fmt(probs['Ace'])} P(Face)={fmt(probs['Face'])} P(Red)={fmt(probs['Red'])}")
print("Most Frequent Category:", most)
print(f"P(Black)={fmt(p_black)}")

Input: cards = ['AH', '10D', 'KC', 'QS', '2H', 'JD', '7C', 'KH']
P(Ace)=0.125 P(Face)=0.50 P(Red)=0.625
Most Frequent Category: Red
P(Black)=0.375


## 32. Toggle K-th Bit

In [32]:
def toggle_bit(n, k):
    return n ^ (1 << k)                             # XOR with a mask that has only bit k set (bit 0 = rightmost)

for n, k in [(5, 1), (4, 2)]:
    r = toggle_bit(n, k)
    print(f"Input: n = {n} ({n:b}), k = {k}  ->  Output: {r} ({r:b})")

Input: n = 5 (101), k = 1  ->  Output: 7 (111)
Input: n = 4 (100), k = 2  ->  Output: 0 (0)


## 33. Plus One

In [33]:
def plus_one(digits):
    digits = digits[:]
    for i in range(len(digits) - 1, -1, -1):
        if digits[i] < 9:
            digits[i] += 1
            return digits
        digits[i] = 0                               # 9 + 1 = 10: write 0 and carry
    return [1] + digits                             # all digits were 9

for d in [[1, 2, 3], [4, 3, 2, 1], [9]]:
    print(f"Input: digits = {d}  ->  Output: {plus_one(d)}")

Input: digits = [1, 2, 3]  ->  Output: [1, 2, 4]
Input: digits = [4, 3, 2, 1]  ->  Output: [4, 3, 2, 2]
Input: digits = [9]  ->  Output: [1, 0]


## 34. String Transformation with Given Substring

In [34]:
def transform(s, sub):
    start = s.find(sub)                             # sub is guaranteed to be in s
    shifted = "".join("z" if ch == "a" else chr(ord(ch) - 1) for ch in sub)
    return s[:start] + shifted + s[start + len(sub):]

for s, sub in [("hackerrank", "hack"), ("programming", "gram"), ("apple", "a")]:
    print(f'Input: s = "{s}", sub = "{sub}"  ->  Output: {transform(s, sub)}')

Input: s = "hackerrank", sub = "hack"  ->  Output: gzbjerrank
Input: s = "programming", sub = "gram"  ->  Output: profqzlming
Input: s = "apple", sub = "a"  ->  Output: zpple


## 35. Trapping Rain Water

In [35]:
def trap(height):
    """Two pointers: the side with the lower wall decides the water level there."""
    left, right = 0, len(height) - 1
    left_max = right_max = water = 0
    while left < right:
        if height[left] < height[right]:
            left_max = max(left_max, height[left])
            water += left_max - height[left]
            left += 1
        else:
            right_max = max(right_max, height[right])
            water += right_max - height[right]
            right -= 1
    return water

for h in [[0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1], [4, 2, 0, 3, 2, 5]]:
    print(f"Input: height = {h}  ->  Output: {trap(h)}")

Input: height = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]  ->  Output: 6
Input: height = [4, 2, 0, 3, 2, 5]  ->  Output: 9


## 36. Generate Parentheses

In [36]:
def generate_parentheses(n):
    result = []
    def backtrack(current, open_count, close_count):
        if len(current) == 2 * n:
            result.append(current); return
        if open_count < n:
            backtrack(current + "(", open_count + 1, close_count)
        if close_count < open_count:
            backtrack(current + ")", open_count, close_count + 1)
    backtrack("", 0, 0)
    return result

for n in [3, 1]:
    print(f"Input: n = {n}  ->  Output: {generate_parentheses(n)}")

Input: n = 3  ->  Output: ['((()))', '(()())', '(())()', '()(())', '()()()']
Input: n = 1  ->  Output: ['()']


## 37. Spiral Matrix

In [37]:
def spiral_order(matrix):
    result = []
    top, bottom, left, right = 0, len(matrix) - 1, 0, len(matrix[0]) - 1
    while top <= bottom and left <= right:
        for j in range(left, right + 1): result.append(matrix[top][j])
        top += 1
        for i in range(top, bottom + 1): result.append(matrix[i][right])
        right -= 1
        if top <= bottom:
            for j in range(right, left - 1, -1): result.append(matrix[bottom][j])
            bottom -= 1
        if left <= right:
            for i in range(bottom, top - 1, -1): result.append(matrix[i][left])
            left += 1
    return result

for m in [[[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]]:
    print(f"Input: matrix = {m}  ->  Output: {spiral_order(m)}")

Input: matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]  ->  Output: [1, 2, 3, 6, 9, 8, 7, 4, 5]
Input: matrix = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]  ->  Output: [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]


## 38. Container With Most Water

In [38]:
def max_area(height):
    left, right, best = 0, len(height) - 1, (0, 0, 0)
    while left < right:
        area = (right - left) * min(height[left], height[right])
        if area > best[0]:
            best = (area, left, right)
        if height[left] < height[right]:            # move the pointer at the lower line
            left += 1
        else:
            right -= 1
    return best

for h in [[1, 8, 6, 2, 5, 4, 8, 3, 7], [1, 1], [4, 3, 2, 1, 4]]:
    area, l, r = max_area(h)
    print(f"Input: height = {h}  ->  Output: {area}  (lines at index {l} and {r})")

Input: height = [1, 8, 6, 2, 5, 4, 8, 3, 7]  ->  Output: 49  (lines at index 1 and 8)
Input: height = [1, 1]  ->  Output: 1  (lines at index 0 and 1)
Input: height = [4, 3, 2, 1, 4]  ->  Output: 16  (lines at index 0 and 4)


## 39. Matrix Chain Multiplication

In [39]:
def matrix_chain(dims):
    n = len(dims) - 1                               # number of matrices
    cost = [[0] * n for _ in range(n)]
    split = [[0] * n for _ in range(n)]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            cost[i][j] = float("inf")
            for k in range(i, j):
                c = cost[i][k] + cost[k + 1][j] + dims[i] * dims[k + 1] * dims[j + 1]
                if c < cost[i][j]:
                    cost[i][j], split[i][j] = c, k

    def paren(i, j):
        if i == j: return f"A{i + 1}"
        return f"({paren(i, split[i][j])} × {paren(split[i][j] + 1, j)})"

    steps = []
    def multiply(i, j):                             # returns (label, rows, cols) of the product
        if i == j: return f"A{i + 1}", dims[i], dims[i + 1]
        k = split[i][j]
        left, right = multiply(i, k), multiply(k + 1, j)
        c = left[1] * left[2] * right[2]
        steps.append(f"({left[0]} × {right[0]}) → cost = {left[1]}×{left[2]}×{right[2]} = {c}")
        return f"Result{len(steps)}", left[1], right[2]
    multiply(0, n - 1)
    return cost[0][n - 1], paren(0, n - 1), steps

for dims in [[10, 30, 5, 60], [40, 20, 30, 10, 30]]:
    total, order, steps = matrix_chain(dims)
    print(f"Input: Dimensions = {dims}")
    print("Minimum Cost:", total)
    print("Optimal Parenthesization:", order)
    print("Multiplication Sequence:")
    for i, s in enumerate(steps, 1): print(f"  Step {i}: {s}")
    print()

Input: Dimensions = [10, 30, 5, 60]
Minimum Cost: 4500
Optimal Parenthesization: ((A1 × A2) × A3)
Multiplication Sequence:
  Step 1: (A1 × A2) → cost = 10×30×5 = 1500
  Step 2: (Result1 × A3) → cost = 10×5×60 = 3000

Input: Dimensions = [40, 20, 30, 10, 30]
Minimum Cost: 26000
Optimal Parenthesization: ((A1 × (A2 × A3)) × A4)
Multiplication Sequence:
  Step 1: (A2 × A3) → cost = 20×30×10 = 6000
  Step 2: (A1 × Result1) → cost = 40×20×10 = 8000
  Step 3: (Result2 × A4) → cost = 40×10×30 = 12000



## 40. Validate and Complete a 9×9 Sudoku Board


In [40]:
def validate_sudoku(board):
    reasons = []
    box_names = ["top-left", "top-middle", "top-right", "middle-left", "centre",
                 "middle-right", "bottom-left", "bottom-middle", "bottom-right"]
    for i in range(9):
        row = [c for c in board[i] if c != "."]
        col = [board[r][i] for r in range(9) if board[r][i] != "."]
        box = [board[r][c] for r in range(3 * (i // 3), 3 * (i // 3) + 3)
               for c in range(3 * (i % 3), 3 * (i % 3) + 3) if board[r][c] != "."]
        for name, vals in [(f"row {i + 1}", row), (f"column {i + 1}", col), (f"the {box_names[i]} 3×3 sub-grid", box)]:
            for d in sorted({v for v in vals if vals.count(v) > 1}):
                reasons.append(f"Duplicate '{d}' found in {name}")
    return not reasons, reasons

def solve_sudoku(board):
    def ok(r, c, d):
        br, bc = 3 * (r // 3), 3 * (c // 3)
        return (all(board[r][j] != d for j in range(9)) and all(board[i][c] != d for i in range(9))
                and all(board[i][j] != d for i in range(br, br + 3) for j in range(bc, bc + 3)))
    for r in range(9):
        for c in range(9):
            if board[r][c] == ".":
                for d in "123456789":
                    if ok(r, c, d):
                        board[r][c] = d
                        if solve_sudoku(board): return True
                        board[r][c] = "."          # backtrack
                return False
    return True

def print_board(board):
    for r in range(9):
        if r in (3, 6): print("-" * 21)
        print(" | ".join(" ".join(board[r][c] for c in range(b, b + 3)) for b in (0, 3, 6)))

valid_board = [["5","3",".",".","7",".",".",".","."],["6",".",".","1","9","5",".",".","."],
               [".","9","8",".",".",".",".","6","."],["8",".",".",".","6",".",".",".","3"],
               ["4",".",".","8",".","3",".",".","1"],["7",".",".",".","2",".",".",".","6"],
               [".","6",".",".",".",".","2","8","."],[".",".",".","4","1","9",".",".","5"],
               [".",".",".",".","8",".",".","7","9"]]
invalid_board = [row[:] for row in valid_board]; invalid_board[0][0] = "8"

for name, board in [("Example 1", valid_board), ("Example 2", invalid_board)]:
    board = [row[:] for row in board]
    ok, reasons = validate_sudoku(board)
    print(f"{name}\nValid Sudoku: {str(ok).lower()}")
    if ok and solve_sudoku(board):
        print("Completed Sudoku:"); print_board(board)
    else:
        for r in reasons: print("Reason:", r)
    print()

Example 1
Valid Sudoku: true
Completed Sudoku:
5 3 4 | 6 7 8 | 9 1 2
6 7 2 | 1 9 5 | 3 4 8
1 9 8 | 3 4 2 | 5 6 7
---------------------
8 5 9 | 7 6 1 | 4 2 3
4 2 6 | 8 5 3 | 7 9 1
7 1 3 | 9 2 4 | 8 5 6
---------------------
9 6 1 | 5 3 7 | 2 8 4
2 8 7 | 4 1 9 | 6 3 5
3 4 5 | 2 8 6 | 1 7 9

Example 2
Valid Sudoku: false
Reason: Duplicate '8' found in column 1
Reason: Duplicate '8' found in the top-left 3×3 sub-grid



## 41. Find All Words in Grid (Trie + Backtracking)

In [41]:
def find_words(board, words):
    trie = {}
    for w in words:                                 # build the trie; "$" marks the end of a word
        node = trie
        for ch in w: node = node.setdefault(ch, {})
        node["$"] = w
    m, n, found = len(board), len(board[0]), set()

    def dfs(r, c, node):
        ch = board[r][c]
        if ch not in node: return
        nxt = node[ch]
        if "$" in nxt: found.add(nxt["$"])
        board[r][c] = "#"                           # mark as used for this path
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and board[nr][nc] != "#":
                dfs(nr, nc, nxt)
        board[r][c] = ch                            # backtrack

    for r in range(m):
        for c in range(n):
            dfs(r, c, trie)
    return sorted(found)

board1 = [["o","a","a","n"], ["e","t","a","e"], ["i","h","k","r"], ["i","f","l","v"]]
print("Example 1 Output:", find_words(board1, ["oath", "pea", "eat", "rain"]))
print("Example 2 Output:", find_words([["a","b"], ["c","d"]], ["abcb"]))

Example 1 Output: ['eat', 'oath']
Example 2 Output: []


## 42. House Robber

In [42]:
def rob(nums):
    take, skip = 0, 0                               # best total if this house is robbed / not robbed
    for x in nums:
        take, skip = skip + x, max(take, skip)
    return max(take, skip)

for nums in [[1, 2, 3, 1], [2, 7, 9, 3, 1]]:
    print(f"Input: nums = {nums}  ->  Output: {rob(nums)}")

Input: nums = [1, 2, 3, 1]  ->  Output: 4
Input: nums = [2, 7, 9, 3, 1]  ->  Output: 12


## 43. Find Euler Path

In [43]:
from collections import defaultdict

def euler_path(n, edges):
    adj = defaultdict(list)
    degree = [0] * n
    for i, (u, v) in enumerate(edges):
        adj[u].append((v, i)); adj[v].append((u, i))
        degree[u] += 1; degree[v] += 1
    odd = [v for v in range(n) if degree[v] % 2]

    # Connectivity check over vertices that have edges
    has_edges = [v for v in range(n) if degree[v] > 0]
    seen, stack = set(), [has_edges[0]] if has_edges else []
    while stack:
        v = stack.pop()
        if v in seen: continue
        seen.add(v); stack.extend(u for u, _ in adj[v])
    connected = all(v in seen for v in has_edges)

    reasons = []
    if len(odd) not in (0, 2): reasons.append("More than 2 nodes have odd degree")
    if not connected: reasons.append("graph is disconnected")
    if reasons:
        return False, degree, odd, None, None, " and ".join(reasons)

    start = odd[0] if odd else has_edges[0]
    for v in adj: adj[v].sort()                     # deterministic order
    used, stack, path = [False] * len(edges), [start], []
    while stack:                                    # Hierholzer's algorithm
        v = stack[-1]
        while adj[v] and used[adj[v][0][1]]: adj[v].pop(0)
        if adj[v]:
            u, i = adj[v].pop(0); used[i] = True; stack.append(u)
        else:
            path.append(stack.pop())
    return True, degree, odd, start, path[::-1], None

for n, edges in [(4, [[0, 1], [1, 2], [2, 0], [1, 3]]), (4, [[0, 1], [1, 2], [2, 3]]), (4, [[0, 1], [2, 3]])]:
    ok, degree, odd, start, path, reason = euler_path(n, edges)
    print(f"Input: n = {n}, edges = {edges}")
    print("Euler Path Exists:", str(ok).lower())
    print("Node Degrees:")
    for v, d in enumerate(degree): print(f"  {v} → {d}")
    print("Odd Degree Nodes:", odd)
    if ok:
        print("Start Node:", start)
        print("Euler Path:", " → ".join(map(str, path)))
    else:
        print("Reason:", reason)
    print()

Input: n = 4, edges = [[0, 1], [1, 2], [2, 0], [1, 3]]
Euler Path Exists: true
Node Degrees:
  0 → 2
  1 → 3
  2 → 2
  3 → 1
Odd Degree Nodes: [1, 3]
Start Node: 1
Euler Path: 1 → 0 → 2 → 1 → 3

Input: n = 4, edges = [[0, 1], [1, 2], [2, 3]]
Euler Path Exists: true
Node Degrees:
  0 → 1
  1 → 2
  2 → 2
  3 → 1
Odd Degree Nodes: [0, 3]
Start Node: 0
Euler Path: 0 → 1 → 2 → 3

Input: n = 4, edges = [[0, 1], [2, 3]]
Euler Path Exists: false
Node Degrees:
  0 → 1
  1 → 1
  2 → 1
  3 → 1
Odd Degree Nodes: [0, 1, 2, 3]
Reason: More than 2 nodes have odd degree and graph is disconnected



## 44. Rotate Image

In [44]:
def rotate(matrix):
    """In place: transpose, then reverse each row = 90° clockwise rotation."""
    n = len(matrix)
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    for row in matrix:
        row.reverse()
    return matrix

for m in [[[1, 2, 3], [4, 5, 6], [7, 8, 9]], [[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]]]:
    print(f"Input: matrix = {m}", end="  ->  ")
    print(f"Output: {rotate(m)}")

Input: matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]  ->  Output: [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
Input: matrix = [[5, 1, 9, 11], [2, 4, 8, 10], [13, 3, 6, 7], [15, 14, 12, 16]]  ->  Output: [[15, 13, 2, 5], [14, 3, 4, 1], [12, 6, 8, 9], [16, 7, 10, 11]]


## 45. Climbing Stairs with Patterns

In [45]:
def climb_stairs(n, list_limit=20):
    a, b = 1, 1                                     # ways(0), ways(1): Fibonacci recurrence
    for _ in range(n - 1):
        a, b = b, a + b
    total = b if n >= 1 else 1
    patterns = []
    if n <= list_limit:                             # listing every pattern explodes for large n
        def backtrack(remaining, steps):
            if remaining == 0:
                patterns.append(" + ".join(map(str, steps))); return
            for s in (1, 2):
                if s <= remaining: backtrack(remaining - s, steps + [s])
        backtrack(n, [])
    return total, patterns

for n in [2, 3, 4]:
    total, patterns = climb_stairs(n)
    print(f"Input: n = {n}\nTotal Ways: {total}\nPatterns:")
    print("\n".join(patterns)); print()

Input: n = 2
Total Ways: 2
Patterns:
1 + 1
2

Input: n = 3
Total Ways: 3
Patterns:
1 + 1 + 1
1 + 2
2 + 1

Input: n = 4
Total Ways: 5
Patterns:
1 + 1 + 1 + 1
1 + 1 + 2
1 + 2 + 1
2 + 1 + 1
2 + 2



## 46. Filter Characters from a String

In [46]:
def filter_chars(target, reference):
    remove = set(reference)                         # set lookup keeps this O(len(target))
    return "".join(ch for ch in target if ch not in remove)

for target, ref in [("programming", "grm"), ("hello world", "lo"), ("abcdef", "xyz")]:
    print(f'Input: target = "{target}", reference = "{ref}"  ->  Output: "{filter_chars(target, ref)}"')

Input: target = "programming", reference = "grm"  ->  Output: "poain"
Input: target = "hello world", reference = "lo"  ->  Output: "he wrd"
Input: target = "abcdef", reference = "xyz"  ->  Output: "abcdef"


## 47. Sum of Odd Numbers in a List

In [47]:
# 1. Using a regular function parameter (the list is passed in and looped over)
def sum_odd_loop(nums):
    total = 0
    for x in nums:
        if x % 2 != 0:
            total += x
    return total

# 2. Using list comprehension
sum_odd_comprehension = lambda nums: sum([x for x in nums if x % 2 != 0])

# 3. Using a user-defined helper function
def is_odd(x):
    return x % 2 != 0

def sum_odd_user_defined(nums):
    return sum(filter(is_odd, nums))

for nums in [[1, 2, 3, 4, 5, 6], [10, 15, 20, 25, 30, 35], [2, 4, 6, 8]]:
    print(f"Input: nums = {nums}")
    print("  Regular function:", sum_odd_loop(nums))
    print("  List comprehension:", sum_odd_comprehension(nums))
    print("  User-defined function:", sum_odd_user_defined(nums))

Input: nums = [1, 2, 3, 4, 5, 6]
  Regular function: 9
  List comprehension: 9
  User-defined function: 9
Input: nums = [10, 15, 20, 25, 30, 35]
  Regular function: 75
  List comprehension: 75
  User-defined function: 75
Input: nums = [2, 4, 6, 8]
  Regular function: 0
  List comprehension: 0
  User-defined function: 0


## 48. Palindrome Number Using a Decorator Function

In [48]:
def palindrome_decorator(func):
    """Adds the display message around a function that only returns the number."""
    def wrapper(num):
        n = func(num)
        if str(n) == str(n)[::-1]:
            print(f"{n} is a Palindrome")
        else:
            print(f"{n} is Not a Palindrome")
    return wrapper

@palindrome_decorator
def check_number(num):
    if not isinstance(num, int) or num < 0:
        raise ValueError("The input must be a non-negative integer.")
    return num

for num in [121, 1234]:
    check_number(num)

121 is a Palindrome
1234 is Not a Palindrome


## 49. Cumulative Average Using a Closure

In [49]:
def cumulative_average():
    numbers = []                                    # remembered between calls by the inner function
    def add_number(num):
        numbers.append(num)
        avg = sum(numbers) / len(numbers)
        print(f"Cumulative Average: {avg}")
        return avg
    return add_number

USE_KEYBOARD = False          # set True to type the numbers yourself

for example in [[10, 20, 30, 40], [5, 15, 25]]:
    average = cumulative_average()
    if USE_KEYBOARD:
        n = int(input("n = "))
        values = [int(input("Enter number: ")) for _ in range(n)]
    else:
        values = example
    print(f"Input: n = {len(values)}, numbers = {values}")
    for v in values:
        average(v)
    print()

Input: n = 4, numbers = [10, 20, 30, 40]
Cumulative Average: 10.0
Cumulative Average: 15.0
Cumulative Average: 20.0
Cumulative Average: 25.0

Input: n = 3, numbers = [5, 15, 25]
Cumulative Average: 5.0
Cumulative Average: 10.0
Cumulative Average: 15.0



## 50. Remove Elements Greater Than 20 Using Decorator and Filter

In [50]:
def report(func):
    """Decorator: prints the input and the result around the filtering function."""
    def wrapper(nums):
        print("Input:", nums)
        result = func(nums)
        print("Output:", result)
        return result
    return wrapper

@report
def remove_greater_than_20(nums):
    return list(filter(lambda x: x <= 20, nums))

filtered = remove_greater_than_20([5, 10, 23, 40, 8])

Input: [5, 10, 23, 40, 8]
Output: [5, 10, 8]


## 51. File and Directory Management Using os and time

In [51]:
import os, time

base = os.path.join("My_Project", "assignments", "task1")

# 1. Create nested directories (no error if they already exist)
os.makedirs(base, exist_ok=True)

# 2. Current working directory, as text and as bytes
print("Current Working Directory:", os.getcwd())
print("Byte String Format:", os.getcwdb())

# 3. Create sample.py and write a statement into it
file_path = os.path.join(base, "sample.py")
with open(file_path, "w") as f:
    f.write("print('Python Assignment')")

# 4. Check that the directory exists
print(os.path.isdir(base))

# 5. Total size of all files under My_Project
total_size = 0
for root, dirs, files in os.walk("My_Project"):
    for name in files:
        total_size += os.path.getsize(os.path.join(root, name))
print(f"total_size: {total_size} bytes")

# 6. Access and modification times in readable form
print("access time:", time.ctime(os.path.getatime(file_path)))
print("Last modified time:", time.ctime(os.path.getmtime(file_path)))

# 7. List the directory contents
print("list of files in the directory:", os.listdir(base))

Current Working Directory: c:\Users\MY.AC.P2MCA25029\Downloads
Byte String Format: b'c:\\Users\\MY.AC.P2MCA25029\\Downloads'
True
total_size: 26 bytes
access time: Thu Nov 12 16:09:52 2026
Last modified time: Thu Nov 12 16:09:52 2026
list of files in the directory: ['sample.py']


## 52. CSV Data Cleaning and JSON Conversion

The PDF shows the CSV on one line because its line breaks were lost; the data here uses one record per line, as a CSV file would.

In [52]:
import csv, json

raw_data = """Name,Age,Salary,Join Date
Alice, 30, 95000, 2015-03-01
Bob, , 60000, 2018-07-15
Charlie, 35, Unknown, 2024-02-01
Diana, 40, 110000, 2021-11-03"""

lines = raw_data.strip().split("\n")                        # 1. split the multiline string into lines
reader = csv.DictReader(lines)                              # 2. read rows as dictionaries

cleaned_records = []
for row in reader:
    row = {k.strip(): (v.strip() if isinstance(v, str) else v) for k, v in row.items()}   # 3. strip spaces
    if not row["Age"] or not row["Salary"] or row["Salary"].lower() == "unknown":       # 4. missing / invalid
        print(f"Skipped {row['Name']}: missing or invalid age/salary")
        continue
    try:
        age, salary = int(row["Age"]), float(row["Salary"])                              # 5. type conversion
    except ValueError:                                                                    # 6. invalid number
        print(f"Skipped {row['Name']}: could not convert age/salary")
        continue
    cleaned_records.append({"name": row["Name"], "age": age,                            # 7-8. new schema
                            "salary": salary, "join_date": row["Join Date"]})

with open("cleaned_data.json", "w") as f:                                               # 9. write JSON
    json.dump(cleaned_records, f, indent=2)

print("\nContents of cleaned_data.json:")
print(open("cleaned_data.json").read())

Skipped Bob: missing or invalid age/salary
Skipped Charlie: missing or invalid age/salary

Contents of cleaned_data.json:
[
  {
    "name": "Alice",
    "age": 30,
    "salary": 95000.0,
    "join_date": "2015-03-01"
  },
  {
    "name": "Diana",
    "age": 40,
    "salary": 110000.0,
    "join_date": "2021-11-03"
  }
]


## 53. Longest Substring Without Repeating Characters (Sliding Window)

In [53]:
def longest_unique_substring(s):
    visited = [False] * 26                          # which letters are in the current window
    left = max_len = 0
    for right in range(len(s)):                     # expand the window
        idx = ord(s[right]) - ord("a")
        while visited[idx]:                         # duplicate: shrink from the left
            visited[ord(s[left]) - ord("a")] = False
            left += 1
        visited[idx] = True
        max_len = max(max_len, right - left + 1)
    return max_len

USE_KEYBOARD = False
inputs = [input("Enter a string: ")] if USE_KEYBOARD else ["abcabcbb", "bbbbb", "pwwkew"]
for s in inputs:
    print(f'Input: s = "{s}"  ->  Length of the longest unique substring: {longest_unique_substring(s)}')

Input: s = "abcabcbb"  ->  Length of the longest unique substring: 3
Input: s = "bbbbb"  ->  Length of the longest unique substring: 1
Input: s = "pwwkew"  ->  Length of the longest unique substring: 3


## 54. Generator Function to Extract Vowels

In [54]:
def getvowels(name):
    vowels = "aeiouAEIOU"
    for ch in name:
        if ch in vowels:
            yield ch                                # returns one vowel at a time

USE_KEYBOARD = False
name = input("Enter name: ") if USE_KEYBOARD else "Priyanka"
print("Enter name:", name)
for v in getvowels(name):
    print(v)

Enter name: Priyanka
i
a
a


## 55. User-Defined Iterator for Decimal Values

In [55]:
class NumberIterator:
    def __init__(self, start=0.0, stop=1.0, step=0.25):
        self.start, self.stop, self.step = start, stop, step
        self.i = 0
    def __iter__(self):
        return self
    def __next__(self):
        value = round(self.start + self.i * self.step, 10)   # computed from the index to avoid float drift
        if value > self.stop:
            raise StopIteration
        self.i += 1
        return value

for value in NumberIterator(0.0, 1.0, 0.25):
    print(value)

0.0
0.25
0.5
0.75
1.0


## 56. Move All Zeros to the End

In [56]:
def move_zeros(l):
    non_zero = [x for x in l if x != 0]             # keeps the original order
    zeros = [x for x in l if x == 0]
    return non_zero + zeros

for l in [[0, 1, 0, 2, 0, 3], [4, 0, 5, 0, 2, 0, 8]]:
    print(f"Input: l = {l}  ->  Output: {move_zeros(l)}")

Input: l = [0, 1, 0, 2, 0, 3]  ->  Output: [1, 2, 3, 0, 0, 0]
Input: l = [4, 0, 5, 0, 2, 0, 8]  ->  Output: [4, 5, 2, 8, 0, 0, 0]


## 57. Student Performance Analyzer

In [57]:
import statistics, random, datetime
from collections import Counter
from functools import reduce

students = {"Tarun": [82, 91, 78], "Karthik": [65, 72, 68], "Chadrika": [91, 95, 89],
            "Trisha": [76, 81, 79], "Atul": [88, 84, 92]}

averages = {name: statistics.mean(marks) for name, marks in students.items()}
print("Student averages:")
for name, avg in averages.items():
    print(f"  {name:9s} {avg:.2f}")

top = max(averages, key=averages.get)
print(f"\nHighest average: {top} ({averages[top]:.2f})")

print("Randomly selected student:", random.choice(list(students)))
print("Current date:", datetime.date.today().strftime("%d-%m-%Y"))

def grade(avg):
    return "A" if avg >= 85 else "B" if avg >= 75 else "C" if avg >= 65 else "D"
grade_counts = Counter(grade(avg) for avg in averages.values())
print("Grade distribution:", dict(sorted(grade_counts.items())))

total = reduce(lambda acc, marks: acc + sum(marks), students.values(), 0)
print("Total marks of all students:", total)

Student averages:
  Tarun     83.67
  Karthik   68.33
  Chadrika  91.67
  Trisha    78.67
  Atul      88.00

Highest average: Chadrika (91.67)
Randomly selected student: Chadrika
Current date: 12-11-2026
Grade distribution: {'A': 2, 'B': 2, 'C': 1}
Total marks of all students: 1231


## 58. Automated Course Project Team Allocation

In [58]:
%%writefile allocation.py
import sys, random, math, datetime
from collections import defaultdict
from itertools import combinations
from functools import reduce

students = [("Anita", "CSE", 85), ("Rahul", "CSE", 78), ("Priya", "CSE", 92), ("Kiran", "CSE", 74),
            ("Meera", "Data Science", 88), ("Arjun", "Data Science", 81), ("Sneha", "Data Science", 90),
            ("Vikram", "AI", 79), ("Divya", "AI", 94), ("Rohan", "AI", 83),
            ("Neha", "Cyber Security", 87), ("Aditya", "Cyber Security", 76)]

# 7. Team size from the command line, e.g.  python allocation.py 3
try:
    team_size = int(sys.argv[1])
except (IndexError, ValueError):
    print("Usage: python allocation.py <team_size>   (using 3)")
    team_size = 3

# 1. Group students by department
by_dept = defaultdict(list)
for name, dept, marks in students:
    by_dept[dept].append((name, marks))

eligible = [d for d, members in by_dept.items() if len(members) >= team_size]
if not eligible:
    sys.exit(f"No department has {team_size} students.")
dept = random.choice(eligible)

# 2. All possible teams in that department, 3. pick one at random
teams = list(combinations(by_dept[dept], team_size))
team = random.sample(teams, 1)[0]

# 4. Total with reduce, 5. average with math
total = reduce(lambda acc, member: acc + member[1], team, 0)
average = math.fsum(m for _, m in team) / team_size

# 6. Allocation date
print("===== PROJECT TEAM ALLOCATION =====")
print("Team Size:", team_size)
print("Allocation Date:", datetime.date.today().strftime("%d-%m-%Y"))
print("Department:", dept)
print(f"Possible teams in this department: {len(teams)}")
print("Selected Team:")
for i, (name, marks) in enumerate(team, 1):
    print(f"{i}. {name} - {marks}")
print("Total Marks:", total)
print(f"Average Marks: {average:.2f}")

Overwriting allocation.py


## 58 (run). Run allocation.py with a team size of 3

In [59]:
import subprocess, sys
print(subprocess.run([sys.executable, "allocation.py", "3"], capture_output=True, text=True).stdout)

===== PROJECT TEAM ALLOCATION =====
Team Size: 3
Allocation Date: 12-11-2026
Department: Data Science
Possible teams in this department: 1
Selected Team:
1. Meera - 88
2. Arjun - 81
3. Sneha - 90
Total Marks: 259
Average Marks: 86.33



## 59. Student Marks Management System

In [60]:
class InvalidMarksError(Exception):
    """Raised when marks are outside 0-100."""

def add_student(name, marks_text, path="student_records.txt"):
    try:
        assert name.strip() != "", "Student name cannot be empty"
        marks = float(marks_text)                    # ValueError for text such as 'ABC'
        if marks < 0 or marks > 100:
            raise InvalidMarksError(f"Marks must be between 0 and 100 (got {marks})")
        with open(path, "a") as f:
            f.write(f"{name.strip()},{marks}\n")
        print(f"Saved: {name.strip()} - {marks}")
    except AssertionError as e:
        print("Assertion error:", e)
    except ValueError:
        print(f"Invalid input: marks must be numeric (got '{marks_text}')")
    except InvalidMarksError as e:
        print("InvalidMarksError:", e)
    finally:
        print("Record processing completed.\n")

USE_KEYBOARD = False
if USE_KEYBOARD:
    while True:
        add_student(input("Student name: "), input("Marks: "))
        if input("Enter another record? (y/n): ").strip().lower() != "y":
            break
else:
    open("student_records.txt", "w").close()        # start with an empty file for the demo
    for name, marks in [("Anita", "85"), ("", "70"), ("Rahul", "ABC"), ("Priya", "105"), ("Kiran", "92")]:
        print(f"Student name: {name!r}  Marks: {marks!r}")
        add_student(name, marks)

print("Contents of student_records.txt:")
print(open("student_records.txt").read())

Student name: 'Anita'  Marks: '85'
Saved: Anita - 85.0
Record processing completed.

Student name: ''  Marks: '70'
Assertion error: Student name cannot be empty
Record processing completed.

Student name: 'Rahul'  Marks: 'ABC'
Invalid input: marks must be numeric (got 'ABC')
Record processing completed.

Student name: 'Priya'  Marks: '105'
InvalidMarksError: Marks must be between 0 and 100 (got 105.0)
Record processing completed.

Student name: 'Kiran'  Marks: '92'
Saved: Kiran - 92.0
Record processing completed.

Contents of student_records.txt:
Anita,85.0
Kiran,92.0



## 60. Insert a Sensor Reading into a NumPy Array

In [61]:
import numpy as np

def insert_reading(arr, position_text, element_text):
    try:
        if not element_text.isdigit():
            raise ValueError(f"'{element_text}' is not an integer reading")
        if not position_text.lstrip("-").isdigit():
            raise ValueError(f"'{position_text}' is not a valid position")
        position = int(position_text)
        if position < 0 or position > len(arr):
            raise IndexError(f"position {position} is outside 0..{len(arr)}")
        arr = np.insert(arr, position, int(element_text))
        print("Updated array:", arr)
    except ValueError as e:
        print("ValueError:", e)
    except IndexError as e:
        print("IndexError:", e)
    finally:
        print("Insertion operation completed.\n")
    return arr

readings = np.array([10, 20, 30, 40, 50])
print("Original array:", readings, "\n")

USE_KEYBOARD = False
cases = [(input("Position: "), input("New reading: "))] if USE_KEYBOARD else \
        [("2", "25"), ("1", "A"), ("3", "Hello"), ("0", "25.5"), ("9", "60")]
for pos, val in cases:
    print(f"Position: {pos}  New reading: {val}")
    readings = insert_reading(readings, pos, val)

Original array: [10 20 30 40 50] 

Position: 2  New reading: 25
Updated array: [10 20 25 30 40 50]
Insertion operation completed.

Position: 1  New reading: A
ValueError: 'A' is not an integer reading
Insertion operation completed.

Position: 3  New reading: Hello
ValueError: 'Hello' is not an integer reading
Insertion operation completed.

Position: 0  New reading: 25.5
ValueError: '25.5' is not an integer reading
Insertion operation completed.

Position: 9  New reading: 60
IndexError: position 9 is outside 0..6
Insertion operation completed.



## 61. Precious Metal Loan Management

In [62]:
RATES = {   # annual simple-interest rate (%) by metal and purity
    "gold": {18: 9.0, 22: 8.5, 24: 8.0},
    "silver": {925: 12.0, 950: 11.5, 999: 11.0},
    "platinum": {900: 10.0, 950: 9.5, 999: 9.0},
}

def calculate_interest(metal, purity, amount, tenure):
    metal = str(metal).strip().lower()
    if metal not in RATES:
        raise ValueError(f"Invalid metal type '{metal}'. Choose Gold, Silver or Platinum.")
    purity = int(str(purity).upper().rstrip("K"))
    if purity not in RATES[metal]:
        raise ValueError(f"Unsupported purity {purity} for {metal}. Allowed: {list(RATES[metal])}")
    rate = RATES[metal][purity]
    return rate, amount * rate * tenure / 100

cases = [("Missing metal type", ()), ("Missing purity", ("gold",)), ("Missing amount", ("gold", 22)),
         ("Missing tenure", ("gold", 22, 500000)), ("Missing multiple arguments", ("silver",)),
         ("Invalid metal", ("diamond", 999, 100000, 2)), ("Invalid purity", ("gold", 15, 100000, 2)),
         ("Valid loan", ("gold", 22, 500000, 3)), ("Valid loan", ("platinum", 950, 200000, 2))]

for label, args in cases:
    print(f"{label}: calculate_interest{args}")
    try:
        rate, interest = calculate_interest(*args)
    except TypeError as e:
        print("  TypeError:", e)
    except ValueError as e:
        print("  ValueError:", e)
    else:
        print(f"  Rate: {rate}%  Interest: ₹{interest:,.2f}  Total payable: ₹{args[2] + interest:,.2f}")
    finally:
        print("  Loan processing step completed.\n")

Missing metal type: calculate_interest()
  TypeError: calculate_interest() missing 4 required positional arguments: 'metal', 'purity', 'amount', and 'tenure'
  Loan processing step completed.

Missing purity: calculate_interest('gold',)
  TypeError: calculate_interest() missing 3 required positional arguments: 'purity', 'amount', and 'tenure'
  Loan processing step completed.

Missing amount: calculate_interest('gold', 22)
  TypeError: calculate_interest() missing 2 required positional arguments: 'amount' and 'tenure'
  Loan processing step completed.

Missing tenure: calculate_interest('gold', 22, 500000)
  TypeError: calculate_interest() missing 1 required positional argument: 'tenure'
  Loan processing step completed.

Missing multiple arguments: calculate_interest('silver',)
  TypeError: calculate_interest() missing 3 required positional arguments: 'purity', 'amount', and 'tenure'
  Loan processing step completed.

Invalid metal: calculate_interest('diamond', 999, 100000, 2)
  Valu

## 62. Rat in a Maze Using Backtracking

In [63]:
def solve_maze(maze):
    n = len(maze)
    path = [[0] * n for _ in range(n)]
    moves = [(1, 0), (0, 1), (-1, 0), (0, -1)]       # down, right, up, left

    def safe(r, c):
        return 0 <= r < n and 0 <= c < n and maze[r][c] == 1 and path[r][c] == 0

    def explore(r, c):
        if (r, c) == (n - 1, n - 1):
            path[r][c] = 1; return True
        if not safe(r, c): return False
        path[r][c] = 1                                # choose
        for dr, dc in moves:                          # explore
            if explore(r + dr, c + dc): return True
        path[r][c] = 0                                # backtrack
        return False

    if maze[0][0] == 1 and maze[n - 1][n - 1] == 1 and explore(0, 0):
        return path
    return None

for maze in [[[1, 0, 0, 0], [1, 1, 0, 1], [0, 1, 0, 0], [1, 1, 1, 1]], [[1, 0, 0], [0, 0, 0], [0, 0, 1]]]:
    print("Input maze:", maze)
    sol = solve_maze(maze)
    if sol:
        print("Solution Path:")
        for row in sol: print(" ".join(map(str, row)))
    else:
        print("No solution exists")
    print()

Input maze: [[1, 0, 0, 0], [1, 1, 0, 1], [0, 1, 0, 0], [1, 1, 1, 1]]
Solution Path:
1 0 0 0
1 1 0 0
0 1 0 0
0 1 1 1

Input maze: [[1, 0, 0], [0, 0, 0], [0, 0, 1]]
No solution exists



## 63. Debugging a Coin-Toss Game

In [64]:
import random

# ---- Buggy version: the player types "Heads", the coin returns "heads" -> never equal ----
def play_buggy(guess, seed):
    random.seed(seed)
    toss = random.choice(["heads", "tails"])
    result = "win" if guess == toss else "lose"
    return toss, result

# ---- Fixed version: normalise the input before comparing ----
def play_fixed(guess, seed, debug=False):
    random.seed(seed)
    guess = guess.strip().lower()
    toss = random.choice(["heads", "tails"])
    if debug:
        breakpoint()     # pdb commands: p guess, p toss, p type(guess), n (next), c (continue)
    result = "win" if guess == toss else "lose"
    assert result == ("win" if guess == toss else "lose")   # logical check
    return toss, result

for seed in range(4):
    toss, buggy = play_buggy("Heads", seed)
    _, fixed = play_fixed("Heads", seed)
    print(f"seed {seed}: toss = {toss:5s} | guess 'Heads' -> buggy: {buggy:4s} fixed: {fixed}")

# To step through with the debugger, run:  play_fixed("Heads", 0, debug=True)

seed 0: toss = tails | guess 'Heads' -> buggy: lose fixed: lose
seed 1: toss = heads | guess 'Heads' -> buggy: lose fixed: win
seed 2: toss = heads | guess 'Heads' -> buggy: lose fixed: win
seed 3: toss = heads | guess 'Heads' -> buggy: lose fixed: win


## 64. Printer Class and Object Identities

In [65]:
class Printer:
    def __init__(self, name, capacity_ppm, color_mode, printer_type):
        self.name, self.capacity_ppm = name, capacity_ppm
        self.color_mode, self.printer_type = color_mode, printer_type
    def __str__(self):
        return f"{self.name}: {self.capacity_ppm} pages/min, {self.color_mode}, {self.printer_type}"

printer1 = Printer("printer1", 30, "Color", "Laser")
printer2 = Printer("printer2", 20, "Monochrome", "Inkjet")
printer3 = Printer("printer3", 45, "Color", "Dot Matrix")

for p in (printer1, printer2, printer3):
    print(p, "| object id:", id(p))

printer1: 30 pages/min, Color, Laser | object id: 2030828314384
printer2: 20 pages/min, Monochrome, Inkjet | object id: 2030828320400
printer3: 45 pages/min, Color, Dot Matrix | object id: 2030828320720


## 65. MovieTicket Class

In [66]:
class MovieTicket:
    def __init__(self, movie_name, theatre, seat_number):
        self.movie_name = movie_name
        self.theatre = theatre
        self.seat_number = seat_number
    def display(self):
        print(f"Movie: {self.movie_name} | Theatre: {self.theatre} | Seat: {self.seat_number}")

ticket1 = MovieTicket("Inception", "PVR Mysuru", "A10")
ticket2 = MovieTicket("Interstellar", "INOX Garuda Mall", "C5")
ticket3 = MovieTicket("Oppenheimer", "DRC Cinemas", "F12")

for t in (ticket1, ticket2, ticket3):
    t.display()

Movie: Inception | Theatre: PVR Mysuru | Seat: A10
Movie: Interstellar | Theatre: INOX Garuda Mall | Seat: C5
Movie: Oppenheimer | Theatre: DRC Cinemas | Seat: F12
